
### Stream Customers Data From Cloud Files to Delta Lake

- Read files from Cloud storage using DataStreamReader API
- Transform the dataframe to add the following columns
    - file_path: cloud_file_path
    - ingestion_date: Current TimeStamp
- write the transformed data stream to Delta Lake Table

#### Read files from Cloud storage using DataStreamReader API

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType

customers_schema = StructType(
    fields=[
        StructField('customer_id', IntegerType()),
        StructField('customer_name', StringType()),
        StructField('date_of_birth', DateType()),
        StructField('telephone', StringType()),
        StructField('email',StringType()),
        StructField('member_since',DateType()),
        StructField('created_timestamp',TimestampType())
    ]
)

In [0]:
customer_df = spark.readStream \
                .format('json') \
                .schema(customers_schema) \
                .load('/Volumes/gizmobox/landing/operational_data/customers_stream')

#### Transform the dataframe to add the following columns
- file_path: cloud_file_path
- ingestion_date: Current TimeStamp

In [0]:
from pyspark.sql.functions import col,current_timestamp

customer_transformed_df = customer_df.withColumn('file_path',col('_metadata.file_path'))\
                                    .withColumn('ingestion_date',current_timestamp())

#### write the transformed data stream to Delta Lake Table

In [0]:
stream_query = customer_transformed_df.writeStream.format('delta')\
    .option('checkpointLocation','/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream')\
    .toTable('gizmobox.bronze.customers_stream')


In [0]:
stream_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customers_stream;

In [0]:
print('hello')